# Project 2, Task 1: Hourly Series, Exploratory Analysis and Forecasting Framework

Project 2 asks to what extent historical charging activity, temporal patterns and other relevant information can
forecast hourly electric-vehicle charging demand over the next 24 hours, for the whole network and for each of the
three districts. In this first task we turn the transaction file into hourly series of session starts, study them,
and fix how forecasts are issued and evaluated before any model is trained.

The data is the original transaction file of the China EV Charging Dataset (Zhang et al., 2025), `Charging_Data.csv`,
with one row per charging session. The notebook runs from the `project2/` folder with Restart and Run All and writes
only to `prepared/`.

Team: Amir Masnavi (1221579), Ines Oliveira (1231205), Manuela Leite (1200720).

## 1 Setup

We fix the seed and the paths, and define three small helpers: one times each section, one saves figures, and one
records each decision with its evidence for the decision log. The data path can be changed through the `P2_DATA`
environment variable, which we use only to check that a cut copy of the file is refused.

In [ ]:
import hashlib
import json
import os
import time
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

SEED = 42
np.random.seed(SEED)

DATA = Path(os.environ.get("P2_DATA", "Charging_Data.csv"))
OUT = Path("prepared")
FIG = OUT / "figures_task1"

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 150

RUN_STARTED = time.perf_counter()
section_times = {}


@contextmanager
def timed(section):
    """Add the wall time of the block to section_times[section] (spec FR-022)."""
    start = time.perf_counter()
    try:
        yield
    finally:
        section_times[section] = round(section_times.get(section, 0.0) + time.perf_counter() - start, 2)


def save_fig(name, fig=None):
    """Save a figure as prepared/figures_task1/<name>.png."""
    FIG.mkdir(parents=True, exist_ok=True)
    (fig or plt.gcf()).savefig(FIG / f"{name}.png", bbox_inches="tight", metadata={"Software": None})


decisions = []


def add_decision(decision_id, section, decision, evidence):
    """Append one entry to the decision log: the team's conclusion, then the evidence behind it."""
    if any(d["id"] == decision_id for d in decisions):
        raise ValueError(f"Decision {decision_id} is already in the log")
    decisions.append({"id": decision_id, "section": section, "decision": decision, "evidence": evidence})


print("data file:", DATA)

## 2 Reading and checking the file

### 2.1 File checks

### 2.2 Test period

### 2.3 Holiday calendar

Public holidays in China, and the weekend days worked in exchange for them, are set by the General Office of the
State Council and published months ahead. Our dates come from three notices: the one on the 2020 holidays (Guobanfa
Mingdian [2019] No. 16), the January 2020 notice that extended the Spring Festival holiday to 2 February, and the one
on the 2021 holidays (Guobanfa Mingdian [2020] No. 27). The dates are therefore known at any issuance time, and we
treat them as known future information.

We type the dates in here instead of adding a package, and we mark the adjusted working days explicitly. These are
weekend days that people work to make a holiday longer, so we expect them to behave like ordinary weekdays. The table
covers 2020 and 2021, test dates included, because the calendar is known in advance and holds no demand data.

In [ ]:
with timed("2 Reading and checking the file"):
    HOLIDAYS = [  # (name, first day, last day)
        ("New Year's Day", "2020-01-01", "2020-01-01"),
        ("Spring Festival", "2020-01-24", "2020-02-02"),
        ("Qingming Festival", "2020-04-04", "2020-04-06"),
        ("Labour Day", "2020-05-01", "2020-05-05"),
        ("Dragon Boat Festival", "2020-06-25", "2020-06-27"),
        ("National Day and Mid-Autumn Festival", "2020-10-01", "2020-10-08"),
        ("New Year's Day", "2021-01-01", "2021-01-03"),
        ("Spring Festival", "2021-02-11", "2021-02-17"),
        ("Qingming Festival", "2021-04-03", "2021-04-05"),
        ("Labour Day", "2021-05-01", "2021-05-05"),
        ("Dragon Boat Festival", "2021-06-12", "2021-06-14"),
        ("Mid-Autumn Festival", "2021-09-19", "2021-09-21"),
        ("National Day", "2021-10-01", "2021-10-07"),
    ]
    ADJUSTED_WORKDAYS = [  # (date, holiday it serves)
        ("2020-01-19", "Spring Festival"),
        ("2020-04-26", "Labour Day"),
        ("2020-05-09", "Labour Day"),
        ("2020-06-28", "Dragon Boat Festival"),
        ("2020-09-27", "National Day and Mid-Autumn Festival"),
        ("2020-10-10", "National Day and Mid-Autumn Festival"),
        ("2021-02-07", "Spring Festival"),
        ("2021-02-20", "Spring Festival"),
        ("2021-04-25", "Labour Day"),
        ("2021-05-08", "Labour Day"),
        ("2021-09-18", "Mid-Autumn Festival"),
        ("2021-09-26", "National Day"),
        ("2021-10-09", "National Day"),
    ]

    days = pd.date_range("2020-01-01", "2021-12-31", freq="D")
    calendar = pd.DataFrame(
        {"day_of_week": days.dayofweek, "is_holiday": False, "is_adjusted_workday": False, "name": ""},
        index=pd.Index(days, name="date"),
    )
    for name, first, last in HOLIDAYS:
        span = pd.date_range(first, last, freq="D")
        assert not calendar.loc[span, "is_holiday"].any(), f"{name} {first} overlaps another holiday"
        calendar.loc[span, "is_holiday"] = True
        calendar.loc[span, "name"] = name
    for day, name in ADJUSTED_WORKDAYS:
        ts = pd.Timestamp(day)
        assert ts.dayofweek >= 5, f"{day} is not a weekend day"
        assert not calendar.at[ts, "is_holiday"], f"{day} is also a holiday"
        calendar.at[ts, "is_adjusted_workday"] = True
        calendar.at[ts, "name"] = f"{name} (adjusted working day)"
    calendar = calendar.reset_index()

    OUT.mkdir(parents=True, exist_ok=True)
    calendar.to_csv(OUT / "task1_calendar.csv", index=False, date_format="%Y-%m-%d")

    by_year = calendar.groupby(calendar["date"].dt.year)[["is_holiday", "is_adjusted_workday"]].sum()
    by_year.index.name = "year"
    display(by_year)
    display(calendar.loc[calendar["name"] != "", ["date", "day_of_week", "name"]].head(12))

    n_holidays = int(calendar["is_holiday"].sum())
    n_adjusted = int(calendar["is_adjusted_workday"].sum())
    add_decision(
        "D02", "2.3",
        "Public holidays and adjusted working days are known future information and enter the calendar as two flags.",
        f"{n_holidays} holiday days and {n_adjusted} adjusted working days in 2020-2021, all published by the State "
        "Council months ahead (notices of 2019 No. 16, 2020 No. 27 and the January 2020 Spring Festival extension).",
    )

## 3 Data-quality audit

## 4 Hourly series

## 5 Exploratory analysis

### 5.1 Level and trend

### 5.2 Daily and weekly seasonality

### 5.3 Autocorrelation and the seasonal period

### 5.4 Holidays

### 5.5 The early-2020 restrictions and change over time

### 5.6 Districts

### 5.7 Weather

## 6 Forecasting formulation

### 6.1 Issuance schedule

### 6.2 Multi-step strategy and forecast rows

### 6.3 Predictor catalogue

### 6.4 A worked issuance

## 7 Evaluation protocol

### 7.1 Test period

### 7.2 Folds

### 7.3 Metrics and the MASE period

### 7.4 Primary score

### 7.5 Baselines and their scores

## 8 Saved files and integrity checks

## 9 Synthesis and decision log